In [1]:
!git clone https://github.com/LiberoBiagi/DL_Nova_IMS_25-26.git

Cloning into 'DL_Nova_IMS_25-26'...
remote: Enumerating objects: 13415, done.
remote: Counting objects: 100% (26/26), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 13415 (delta 14), reused 0 (delta 0), pack-reused 13389 (from 2)
Receiving objects: 100% (13415/13415), 716.68 MiB | 19.31 MiB/s, done.
Resolving deltas: 100% (21/21), done.
Updating files: 100% (13351/13351), done.


In [2]:
%cd DL_Nova_IMS_25-26

/content/DL_Nova_IMS_25-26


## 1. Imports

In [3]:
from preprocessing_functions import *

# model building
from keras import Model
from keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.models import Sequential
from keras.layers import GlobalAveragePooling2D

# model training imports
from keras.optimizers import SGD, Adam
from keras.losses import CategoricalCrossentropy
from keras.metrics import CategoricalAccuracy, AUC, F1Score
from keras.callbacks import ModelCheckpoint, CSVLogger, LearningRateScheduler

## 2. Repeating the preprocessing steps

In [4]:
# load the split files
train_df = pd.read_csv("splits/train.csv")
val_df = pd.read_csv("splits/val.csv")
test_df = pd.read_csv("splits/test.csv")

train_ds, val_ds, test_ds, data_augmentation = preprocess_v1(train_df, val_df, test_df)

## 3. Models

Check if input shapes are correct and if pixel values are in the expected range (just to be sure that the preprocessing is working as intended).

In [5]:
# checking one bacth of tarining images and labels
for img, label in train_ds.take(1):
    print("Shape:", img.shape) # should be (nº batches, height, width, 3) --> 3 color channels
    print("Min pixel:", tf.reduce_min(img).numpy())
    print("Max pixel:", tf.reduce_max(img).numpy())
    print("Label:", label.numpy())

Shape: (64, 512, 512, 3)
Min pixel: 0.0
Max pixel: 1.0
Label: [14 18  2 21 18 14 15 17 14 11 16  5  0 17 17  5  6  5  8  7 10  1 13  1
 15 22 17 14 10  9  8 18 17 14 14  3 22 15 13 22  3 20 13 17 22  3  5  4
  4 13 17 15 20 16  6  4 11 14 12  1 14  0  7 11]


In [6]:
# Baseline model

input_shape = (512, 512, 3)
num_classes = 23
batch_size = 16
epochs = 20

def baseline_cnn(input_shape=(512, 512, 3), num_classes=23):
    model = Sequential([
        Input(shape=input_shape),
        data_augmentation,

        Conv2D(32, (3,3), activation='relu', padding='same'),  # finds patterns in the images, padding='same' to avoid loosing information at the borders (size not reduced)
        MaxPooling2D((2,2), padding='same'),  # reduces the spatial dimensions by half (size reduced to 256x256)
                                              # padding='same' to avoid erros when the input size cannot be divided by 2

        Conv2D(64, (3,3), activation='relu', padding='same'),  # finds more complex patterns
        MaxPooling2D((2,2), padding='same'),  # size reduced to 128x128

        Conv2D(128, (3,3), activation='relu', padding='same'),  # finds even more complex patterns
        MaxPooling2D((2,2), padding='same'),  # size reduced to 64x64

        GlobalAveragePooling2D(),  # 128*256

        Dense(256, activation='relu'),   # fully connected dense layer, takes all the features and combines them
        Dropout(0.5),                    # randomly disactivates 50% of neurons during training
        Dense(num_classes, activation='softmax')  # output layer
    ])
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'
        ]
    )
    return model

In [7]:
baseline_model= baseline_cnn()
baseline_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential (Sequential)         │ (None, 512, 512, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 512, 512, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 256, 256, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 256, 256, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 128, 128, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 128, 128, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 64, 64, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 23)             │         5,911 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 132,183 (516.34 KB)

 Trainable params: 132,183 (516.34 KB)

 Non-trainable params: 0 (0.00 B)

In [8]:
history = baseline_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=epochs
)

Epoch 1/20
169/169 ━━━━━━━━━━━━━━━━━━━━ 109s 560ms/step - accuracy: 0.1454 - loss: 2.9434 - val_accuracy: 0.2140 - val_loss: 2.7289
Epoch 2/20
 80/169 ━━━━━━━━━━━━━━━━━━━━ 46s 518ms/step - accuracy: 0.2138 - loss: 2.7592

KeyboardInterrupt: 

In [ ]:
import tensorflow as tf

print(tf.config.list_physical_devices('GPU'))